# Person 2 — A7 model benchmarks (Colab GPU)
Run cells in order. Choose **Runtime → Change runtime type → GPU** first.
Four assigned models × the team's unchanged 25 prompts = **100 runs**.
This notebook downloads model weights in Colab, never on your laptop. No API key is needed for these public models.

The explicit condition is CUDA, NF4 weights with FP32 compute/nonquantized layers, eager attention, batch 1, greedy decoding, 220 new tokens. Record this condition alongside the results: it differs from native precision teammates may use. Throughput is single-request generation speed, not proven production capacity. Model download/loading time is excluded. Actual GPU compatibility and outputs must be verified by this run; local checks used mocks only.

Keep raw errors. Do not count failed rows as completed benchmarks or treat blank review scores as final scores.

In [1]:
from google.colab import files
uploaded = files.upload()  # Select hestia-a7-person2-colab.zip only.
import zipfile
from pathlib import Path
import io
assert len(uploaded) == 1, "Select only the Colab bundle ZIP."
bundle_name, bundle_data = next(iter(uploaded.items()))
assert bundle_name.lower().endswith(".zip"), "Upload the Colab bundle ZIP."
with zipfile.ZipFile(io.BytesIO(bundle_data)) as archive:
    archive.extractall("/content")
%cd /content/hestia-a7-person2

Saving hestia-a7-person2-colab.zip to hestia-a7-person2-colab.zip
/content/hestia-a7-person2


In [2]:
%pip install -r a7/requirements-person2-colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 140.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 145.4 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [bitsandbytes]
ERROR: pip's dependency resolver does not currently take into account all the packages tha

In [3]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before proceeding."
print(torch.cuda.get_device_name(0))
print("GPU GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
!nvidia-smi
!python -m a7.benchmark_runner --owner P2 --dry-run

Tesla T4
GPU GiB: 14.56317138671875
Sat Oct 24 04:59:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------

## Download weights and run the matrix
The next cell automatically downloads weights through Transformers, loads one model at a time, and checkpoints the CSV after each response. It can take a long time. Rerunning the cell resumes successful rows and retries failures; a copy of the prior CSV is retained first. Never combine runs with different precision, prompt files, packages or GPU conditions. Download partial results before disconnecting; Colab's local storage is temporary.

In [4]:
import hashlib, importlib.metadata, json, platform, shutil
from datetime import datetime, timezone
from a7.benchmark_runner import run_matrix, select_models
out = Path("a7/results/person2_nf4_colab")
out.mkdir(parents=True, exist_ok=True)
csv_path = out / "person2_benchmark.csv"
metadata_path = out / "person2_benchmark.environment.json"
condition = {
    "models": [m["model_id"] for m in select_models(owner="P2")],
    "gpu": torch.cuda.get_device_name(0), "gpu_bytes": torch.cuda.get_device_properties(0).total_memory,
    "python": platform.python_version(), "device": "cuda", "quantization": "nf4",
    "compute_dtype": "float32", "attention": "eager", "max_new_tokens": 220,
    "do_sample": False, "batch_size": 1,
    "packages": {p: importlib.metadata.version(p) for p in ["torch", "transformers", "accelerate", "bitsandbytes"]},
    "source_sha256": {p: hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in [
        "a7/benchmark_runner.py", "a7/benchmark_contract.py", "a7/model_registry.csv", "a7/prompts.csv"]},
}
if metadata_path.exists():
    assert json.loads(metadata_path.read_text())["condition"] == condition, "Changed condition: choose a new results directory."
elif csv_path.exists():
    raise RuntimeError("CSV has no condition metadata. Preserve it and choose a new results directory.")
else:
    metadata_path.write_text(json.dumps({"started_utc": datetime.now(timezone.utc).isoformat(), "condition": condition}, indent=2))
if csv_path.exists():
    shutil.copy2(csv_path, out / ("before_resume_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f") + ".csv"))
rows = run_matrix(select_models(owner="P2"), csv_path, device_request="cuda",
                  dtype_request="float32", quantization="nf4", max_new_tokens=220)
print("Saved", len(rows), "rows to", csv_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Saved 100 rows to a7/results/person2_nf4_colab/person2_benchmark.csv


In [5]:
import pandas as pd
from a7.benchmark_runner import validate_result_rows
from a7.benchmark_contract import PROMPTS
frame = pd.read_csv(csv_path, keep_default_na=False)
validate_result_rows(frame.to_dict("records"), select_models(owner="P2"), PROMPTS)
display(frame.groupby(["model_id", "status"]).size().rename("rows").to_frame())
display(frame.loc[frame.status != "ok", ["model_id", "prompt_id", "error"]])
print("All 100 succeeded:", len(frame) == 100 and frame.status.eq("ok").all())

,,rows
model_id,status,
HuggingFaceTB/SmolLM2-1.7B-Instruct,ok,25
Qwen/Qwen2-0.5B-Instruct,ok,25
facebook/opt-125m,ok,25
mistralai/Mistral-7B-Instruct-v0.3,ok,25


,model_id,prompt_id,error


All 100 succeeded: True


## Review and save evidence
Read every generated response against `a7/evaluation_rubric.csv`; fill the five score components and the total (1–10), plus concrete review notes. An automated first pass is not human evaluation. Retain this raw CSV and create a separately reviewed copy. Save the executed notebook via **File → Download → Download .ipynb**. Send the ZIP and notebook back for validation, cost analysis and final handoff. Do not include downloaded model weights.

In [6]:
import shutil
archive_path = shutil.make_archive("/content/person2_benchmark_evidence", "zip", root_dir=out)
files.download(archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>